In [ ]:
# === System & OS Utilities ===
import os
import math
import warnings
import logging
warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)

# === Progress Bar ===
from tqdm import tqdm

# === TensorFlow & Keras ===
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers
from tensorflow.keras import Model, Input

import os
import numpy as np
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# === Data Handling ===
import numpy as np
import pandas as pd

# === Visualization ===
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# === Scikit-learn ===
from sklearn.model_selection import train_test_split

from scipy import stats
from scipy.ndimage import gaussian_filter

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPUs detected: {len(gpus)}")
    except RuntimeError as e:
        print(f"Error setting GPU memory growth: {e}")
else:
    print("No GPUs detected.")

In [ ]:
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
EPOCH = 100
img_shape=(*IMG_SIZE,3)
dataset_path = r"preprocessed_breakhis_dataset/tissue_only_normalized"
labels = ['Benign', 'Malignant']
print("Class Names:", labels)

In [ ]:
def data_frame(root_path, target_folder):

    class_paths = []
    classes = []
    patient_ids = []

    for label in os.listdir(root_path):

        class_dir = os.path.join(root_path, label)
        if not os.path.isdir(class_dir):
            continue

        target_dir = os.path.join(class_dir, target_folder)

        if not os.path.isdir(target_dir):
            print(f"[WARNING] '{target_folder}' not found in class '{label}'. Skipping.")
            continue

        for image in os.listdir(target_dir):

            if not image.lower().endswith(('.png','.jpg','.jpeg')):
                continue

            path = os.path.join(target_dir,image)

            parts = image.split('-')

            # Skip malformed filenames
            if len(parts) < 3:
                print(f"Skipping malformed filename: {image}")
                continue

            patient_id = parts[1] + "-" + parts[2]

            class_paths.append(path)
            classes.append(label)
            patient_ids.append(patient_id)

    df = pd.DataFrame({
        'Class Path': class_paths,
        'Class': classes,
        'PatientID': patient_ids
    })

    return df

In [ ]:
main_frame = data_frame(
    root_path=dataset_path,
    target_folder="400X"   # "40X", "100X", "200X", "400X"
)

patients = main_frame['PatientID'].unique()

train_pat, temp_pat = train_test_split(
    patients,
    test_size=0.20,
    random_state=42
)

val_pat, test_pat = train_test_split(
    temp_pat,
    test_size=0.5,
    random_state=42
)

train_dataframe = main_frame[main_frame['PatientID'].isin(train_pat)]
validation_dataframe = main_frame[main_frame['PatientID'].isin(val_pat)]
test_dataframe = main_frame[main_frame['PatientID'].isin(test_pat)]


print("Train patients:",len(set(train_dataframe.PatientID)))
print("Val patients:",len(set(validation_dataframe.PatientID)))
print("Test patients:",len(set(test_dataframe.PatientID)))

print("Overlap train-val:",set(train_dataframe.PatientID) & set(validation_dataframe.PatientID))
print("Overlap train-test:",set(train_dataframe.PatientID) & set(test_dataframe.PatientID))
print("Overlap val-test:",set(validation_dataframe.PatientID) & set(test_dataframe.PatientID))
print()
train_counts = train_dataframe['Class'].value_counts()
val_counts = validation_dataframe['Class'].value_counts()
test_counts = test_dataframe['Class'].value_counts()

count_table = pd.DataFrame({
    "Train": train_counts,
    "Validation": val_counts,
    "Test": test_counts
}).fillna(0).astype(int)

count_table["Total"] = count_table.sum(axis=1)
grand_total = pd.DataFrame({
    "Train": [count_table["Train"].sum()],
    "Validation": [count_table["Validation"].sum()],
    "Test": [count_table["Test"].sum()],
    "Total": [count_table["Total"].sum()]
}, index=["Total"])
count_table = pd.concat([count_table, grand_total])

print(count_table)

BATCH_SIZE = min(BATCH_SIZE, len(validation_dataframe))
print(f"Batch Size = {BATCH_SIZE}")

In [ ]:
def preprocess_image(image):

    image = image.astype(np.float32)

    min_val = image.min()
    max_val = image.max()

    # avoid division by zero
    if max_val - min_val < 1e-6:
        return np.zeros_like(image, dtype=np.float32)

    # min-max scale to [0, 1]
    image = (image - min_val) / (max_val - min_val)
    
    return image

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_image,

    # ---- Geometric ----
    rotation_range=30,         
    zoom_range=0.1,           
    width_shift_range=0.05,
    height_shift_range=0.05,
    shear_range=0.05,

    horizontal_flip=True,
    vertical_flip=True,

    fill_mode="reflect"
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_image)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_image)


train_data = train_datagen.flow_from_dataframe(train_dataframe, x_col='Class Path',
                                  y_col='Class', batch_size=BATCH_SIZE,
                                  target_size=IMG_SIZE, class_mode='binary', seed=42)

validation_data = val_datagen.flow_from_dataframe(validation_dataframe, x_col='Class Path',
                                     y_col='Class', batch_size=BATCH_SIZE,
                                     target_size=IMG_SIZE, class_mode='binary', seed=42)

test_data = test_datagen.flow_from_dataframe(test_dataframe, x_col='Class Path',
                                  y_col='Class', batch_size=BATCH_SIZE,
                                  target_size=IMG_SIZE, class_mode='binary', shuffle=False, seed=42)

In [ ]:
class RestoreBestValidationModel(Callback):
    """
    Custom Keras Callback to restore model weights from the best epoch based on validation accuracy.
    - Tracks the best validation accuracy during training.
    - Restores the model weights from the epoch with the best validation accuracy when training finishes.
    """
    
    def __init__(self):
        super(RestoreBestValidationModel, self).__init__()
        # Initialize best validation accuracy, epoch, and weights to track the best model
        self.best_val_acc = -1
        self.best_epoch = -1
        self.best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        """
        Callback function executed at the end of each epoch.
        - Compares the current validation accuracy with the best validation accuracy.
        - If the current validation accuracy is higher or tied but at a later epoch, updates the best model weights.
        """
        # Get validation accuracy from logs
        val_acc = logs.get('val_accuracy')
        
        if val_acc is not None:
            # Update the best validation accuracy and weights if the current one is better or at a later epoch
            if val_acc > self.best_val_acc or (val_acc == self.best_val_acc and epoch > self.best_epoch):
                self.best_val_acc = val_acc
                self.best_epoch = epoch
                self.best_weights = self.model.get_weights()  # Store the model's weights
                print(f"\nModel weights updated at epoch {epoch + 1} with val_accuracy: {val_acc:.4f}")

    def on_train_end(self, logs=None):
        """
        Callback function executed at the end of training.
        - Restores the model weights from the epoch with the best validation accuracy.
        """
        # If we found a better model during training, restore the best weights
        if self.best_weights is not None:
            print(f"Restoring model weights from best epoch {self.best_epoch + 1} with val_accuracy: {self.best_val_acc:.4f}")
            self.model.set_weights(self.best_weights)  # Set the weights back to the best model's weights

# Instantiate the callback
restore_best = RestoreBestValidationModel()


In [ ]:
######## Main Model which is used in this study of breast cancer###########
# ---------------------------
# Squeeze-and-Excitation
# ---------------------------
@tf.keras.utils.register_keras_serializable()
class SEBlock(layers.Layer):
    def __init__(self, reduction=8, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        C = input_shape[-1]
        self.gap = layers.GlobalAveragePooling2D()
        self.fc1 = layers.Dense(C // self.reduction, activation="relu")
        self.fc2 = layers.Dense(C, activation="sigmoid")

    def call(self, x):
        s = self.gap(x)
        s = self.fc1(s)
        s = self.fc2(s)
        s = tf.reshape(s, (-1, 1, 1, x.shape[-1]))
        return x * s


# ---------------------------
# CoordAttention
# ---------------------------
@tf.keras.utils.register_keras_serializable()
class CoordAttention(layers.Layer):
    def __init__(self, reduction=32, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        C = input_shape[-1]
        mip = max(8, C // self.reduction)
        self.conv1 = layers.Conv2D(mip, 1, activation="relu")
        self.conv_h = layers.Conv2D(C, 1)
        self.conv_w = layers.Conv2D(C, 1)

    def call(self, x):
        h = tf.reduce_mean(x, axis=2, keepdims=True)
        w = tf.reduce_mean(x, axis=1, keepdims=True)
        w = tf.transpose(w, [0, 2, 1, 3])

        y = tf.concat([h, w], axis=1)
        y = self.conv1(y)

        h, w = tf.split(y, [tf.shape(h)[1], tf.shape(w)[1]], axis=1)
        w = tf.transpose(w, [0, 2, 1, 3])

        ah = tf.sigmoid(self.conv_h(h))
        aw = tf.sigmoid(self.conv_w(w))
        return x * ah * aw


# ---------------------------
# Ghost Module
# ---------------------------
def GhostModule(x, out_ch, ratio=2):
    init_ch = math.ceil(out_ch / ratio)

    y = layers.Conv2D(init_ch, 1, use_bias=False)(x)
    y = layers.BatchNormalization()(y)
    y = layers.Activation("relu")(y)

    # kernel 3 → 5 (capacity ↑)
    ghost = layers.DepthwiseConv2D(5, padding="same", use_bias=False)(y)
    ghost = layers.BatchNormalization()(ghost)
    ghost = layers.Activation("relu")(ghost)

    y = layers.Concatenate()([y, ghost])
    y = layers.Conv2D(out_ch, 1, use_bias=False)(y)
    y = layers.BatchNormalization()(y)
    return y


# ---------------------------
# Residual Block 
# ---------------------------
def ResidualBlock(x, out_ch, attention=None, drop_rate=0.1):
    shortcut = x

    y = GhostModule(x, out_ch)
    y = layers.Activation("relu")(y)

    if attention == "se":
        y = SEBlock()(y)
    elif attention == "coord":
        y = CoordAttention()(y)

    # spatial dropout stabilizes feature maps
    y = layers.SpatialDropout2D(drop_rate)(y)

    if x.shape[-1] != out_ch:
        shortcut = layers.Conv2D(out_ch, 1, use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    return layers.Activation("relu")(layers.Add()([shortcut, y]))


# ---------------------------
# Build FINAL Model 
# ---------------------------
def build_skin_model(input_shape=(224,224,3)):
    inp = Input(shape=input_shape)

    # Stem: kernel 3 → 5
    x = layers.Conv2D(32, 5, strides=2, padding="same", use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    # Stage 1
    x = ResidualBlock(x, 48, drop_rate=0.05)
    x = ResidualBlock(x, 48, drop_rate=0.05)

    # Stage 2
    x = layers.Conv2D(96, 5, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = ResidualBlock(x, 96, attention="se", drop_rate=0.1)
    x = ResidualBlock(x, 96, attention="se", drop_rate=0.1)

    # Stage 3
    x = layers.Conv2D(160, 5, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = ResidualBlock(x, 160, attention="coord", drop_rate=0.15)
    x = ResidualBlock(x, 160, attention="coord", drop_rate=0.15)
    x = ResidualBlock(x, 160, attention="coord", drop_rate=0.15)

    # Stage 4
    x = layers.Conv2D(224, 5, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = ResidualBlock(x, 224, attention="coord", drop_rate=0.2)

    # Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(
        256,
        activation="relu",
        kernel_regularizer=tf.keras.regularizers.l1_l2(l1=1e-5, l2=1e-4)
    )(x)
    x = layers.Dropout(0.45)(x)
    x = layers.Dense(1, activation="sigmoid", bias_initializer="zeros")(x)

    return Model(inp, x, name="LightWeightBreastCancerDetector")


# ---------------------------
# Build & Summary
# ---------------------------
model = build_skin_model()
#model.summary()

In [ ]:
model = load_model('Proposed_Model.keras')  

In [ ]:
# ================= Grad-CAM++ Class =================
class GradCAMPlusPlus:
    def __init__(self, model, target_layer, labels):
        self.model = model
        self.labels = labels
        try:
            self.target_layer = model.get_layer(target_layer)
        except:
            print(f"❌ Layer '{target_layer}' not found!")
            print(f"Available Conv layers: {[l.name for l in model.layers if 'Conv' in l.__class__.__name__]}")
            raise
        print(f"✓ Grad-CAM++ initialized with layer: {target_layer}")

    def compute_gradcam_plus_plus(self, image, class_idx):
        img_tensor = tf.expand_dims(tf.convert_to_tensor(image, dtype=tf.float32), 0)
        intermediate_model = Model(inputs=self.model.input, outputs=[self.target_layer.output, self.model.output])
        with tf.GradientTape() as tape:
            layer_output, predictions = intermediate_model(img_tensor)
            tape.watch(layer_output)
            loss = predictions[0, class_idx]
        gradients = tape.gradient(loss, layer_output)
        layer_output_np = layer_output.numpy()[0]
        gradients_np = gradients.numpy()[0]
        second_deriv = np.power(gradients_np, 2)
        third_deriv = second_deriv * gradients_np
        alpha_denom = 2 * second_deriv + np.sum(third_deriv * layer_output_np, axis=(0, 1), keepdims=True)
        alpha_denom = np.where(alpha_denom != 0, alpha_denom, 1e-8)
        alpha = second_deriv / alpha_denom
        alpha = np.maximum(alpha, 0)
        weights = np.sum(alpha * np.maximum(gradients_np, 0), axis=(0, 1))
        weights = weights / (np.sum(weights) + 1e-8)
        cam = np.zeros(layer_output_np.shape[:2])
        for i, w in enumerate(weights):
            cam += w * layer_output_np[:, :, i]
        cam = np.maximum(cam, 0)
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

    def predict_and_explain(self, image):
        img_tensor = tf.expand_dims(tf.convert_to_tensor(image, dtype=tf.float32), 0)
        preds = self.model.predict(img_tensor, verbose=0)[0]
        pred_idx = int(np.argmax(preds))
        confidence = float(preds[pred_idx])
        gradcam = self.compute_gradcam_plus_plus(image, pred_idx)
        gradcam_resized = tf.image.resize(tf.expand_dims(gradcam, -1), image.shape[:2]).numpy()[..., 0]
        return {'prediction': self.labels[pred_idx], 'confidence': confidence, 'gradcam': gradcam_resized, 'preds': preds}

    def visualize_gradcam_plus_plus(self, image, figsize=(14, 4)):
        result = self.predict_and_explain(image)
        fig, axes = plt.subplots(1, 3, figsize=figsize)
        fig.suptitle(f'Grad-CAM++: {result["prediction"]} ({result["confidence"]:.1%})', fontsize=14, fontweight='bold')
        # Original image
        axes[0].imshow(np.clip(image*255,0,255).astype(np.uint8)); axes[0].set_title('Original'); axes[0].axis('off')
        # Heatmap overlay
        axes[1].imshow(np.clip(image*255,0,255).astype(np.uint8), alpha=1.0)
        im = axes[1].imshow(result['gradcam'], cmap='jet', alpha=0.5)
        axes[1].set_title('Grad-CAM++ Heatmap'); axes[1].axis('off'); plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        # Blended overlay
        img_norm = image/255.0 if image.max()>1 else image
        heatmap_rgb = plt.cm.jet(result['gradcam'])[..., :3]
        alpha_blend = result['gradcam'][..., np.newaxis]
        overlay = (1-alpha_blend)*img_norm + alpha_blend*heatmap_rgb
        overlay = np.clip(overlay, 0, 1)
        axes[2].imshow(overlay); axes[2].set_title('Blended Overlay'); axes[2].axis('off')
        plt.tight_layout()
        return fig

In [ ]:
# ================= Example Usage =================
target_layer = 'conv2d_23'    #'conv2d_72'  
image = test_data[0][0][0]
print(image.shape)
explainer = GradCAMPlusPlus(model, target_layer, labels)
fig = explainer.visualize_gradcam_plus_plus(image)
plt.show()

In [ ]:
# ================= Score-CAM Class =================
class ScoreCAM:
    def __init__(self, model, target_layer, labels, max_channels=32):
        self.model = model
        self.labels = labels
        self.max_channels = max_channels
        try:
            self.target_layer = model.get_layer(target_layer)
        except:
            print(f"❌ Layer '{target_layer}' not found!")
            print(f"Available Conv layers: {[l.name for l in model.layers if 'Conv' in l.__class__.__name__]}")
            raise
        print(f"✓ Score-CAM initialized with layer: {target_layer}")

    def compute_scorecam(self, image, class_idx):
        img_tensor = tf.expand_dims(tf.convert_to_tensor(image, dtype=tf.float32), 0)

        # Model for feature maps
        feature_model = Model(self.model.input, self.target_layer.output)
        feature_maps = feature_model(img_tensor)[0]  # (H, W, C)

        num_channels = min(feature_maps.shape[-1], self.max_channels)
        cam = np.zeros(feature_maps.shape[:2], dtype=np.float32)

        for i in range(num_channels):
            fmap = feature_maps[..., i]

            # Normalize activation map
            fmap = np.maximum(fmap, 0)
            if fmap.max() == 0:
                continue
            fmap_norm = fmap / (fmap.max() + 1e-8)

            # Resize to input size
            fmap_resized = tf.image.resize(
                fmap_norm[..., np.newaxis], image.shape[:2]
            ).numpy()

            # Mask input image
            masked_img = image * fmap_resized

            # Get class score
            masked_tensor = tf.expand_dims(masked_img, 0)
            score = self.model(masked_tensor, training=False)[0, class_idx]

            cam += score.numpy() * fmap_norm

        cam = np.maximum(cam, 0)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

    def predict_and_explain(self, image):
        img_tensor = tf.expand_dims(tf.convert_to_tensor(image, dtype=tf.float32), 0)
        preds = self.model.predict(img_tensor, verbose=0)[0]
        pred_idx = int(np.argmax(preds))
        confidence = float(preds[pred_idx])

        scorecam = self.compute_scorecam(image, pred_idx)
        scorecam_resized = tf.image.resize(
            scorecam[..., np.newaxis], image.shape[:2]
        ).numpy()[..., 0]

        return {
            'prediction': self.labels[pred_idx],
            'confidence': confidence,
            'scorecam': scorecam_resized,
            'preds': preds
        }

    def visualize_scorecam(self, image, figsize=(14, 4)):
        result = self.predict_and_explain(image)

        fig, axes = plt.subplots(1, 3, figsize=figsize)
        fig.suptitle(
            f'Score-CAM: {result["prediction"]} ({result["confidence"]:.1%})',
            fontsize=14, fontweight='bold'
        )

        # Original image
        axes[0].imshow(np.clip(image*255,0,255).astype(np.uint8))
        axes[0].set_title('Original')
        axes[0].axis('off')

        # Heatmap overlay
        axes[1].imshow(np.clip(image*255,0,255).astype(np.uint8))
        im = axes[1].imshow(result['scorecam'], cmap='jet', alpha=0.5)
        axes[1].set_title('Score-CAM Heatmap')
        axes[1].axis('off')
        plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

        # Blended overlay
        img_norm = image/255.0 if image.max()>1 else image
        heatmap_rgb = plt.cm.jet(result['scorecam'])[..., :3]
        alpha = result['scorecam'][..., np.newaxis]
        overlay = (1-alpha)*img_norm + alpha*heatmap_rgb
        overlay = np.clip(overlay, 0, 1)

        axes[2].imshow(overlay)
        axes[2].set_title('Blended Overlay')
        axes[2].axis('off')

        plt.tight_layout()
        return fig


In [ ]:
target_layer = 'conv2d_23'
image = test_data[0][0][0]

explainer = ScoreCAM(model, target_layer, labels)
fig = explainer.visualize_scorecam(image)
plt.show()

In [ ]:
# # ============================================================
# #  Statistical Validation for GradCAM++ vs ScoreCAM
# #  Drop-in — does NOT modify your existing classes at all
# # ============================================================
# from scipy import stats
# from scipy.ndimage import gaussian_filter
# import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.gridspec as gridspec
# from tqdm import tqdm

# # ── reuse your existing helpers ──────────────────────────────
# def ensure_float01(img):
#     img = np.array(img, dtype=np.float32)
#     return img / 255.0 if img.max() > 1.0 else img

# def upsample_cam(cam, target_shape):
#     cam_tf = tf.image.resize(cam[..., np.newaxis],
#                              target_shape[:2], method='bilinear').numpy()[..., 0]
#     mn, mx = cam_tf.min(), cam_tf.max()
#     return (cam_tf - mn) / (mx - mn + 1e-8) if mx > mn else np.zeros_like(cam_tf)

# def deletion_insertion(model, image, cam, class_idx,
#                        steps=30, mode='deletion', baseline='blur'):
#     img  = ensure_float01(image)
#     H, W, _ = img.shape
#     order    = np.argsort(-cam.flatten())
#     base     = gaussian_filter(img, sigma=(7,7,0)) if baseline == 'blur' else np.zeros_like(img)
#     step_px  = max(1, H*W // steps)
#     scores   = []

#     cur = img.copy() if mode == 'deletion' else base.copy()
#     scores.append(float(model.predict(cur[np.newaxis], verbose=0)[0][class_idx]))

#     for s in range(1, steps + 1):
#         k   = min(s * step_px, H*W)
#         idx = order[:k]
#         ys, xs = idx // W, idx % W
#         if mode == 'deletion':
#             cur[ys, xs, :] = base[ys, xs, :]
#         else:
#             cur[ys, xs, :] = img[ys, xs, :]
#         scores.append(float(model.predict(cur[np.newaxis], verbose=0)[0][class_idx]))

#     xs_axis = np.linspace(0, 1, len(scores))
#     auc = float(np.trapz(scores, xs_axis))   # raw area (0–1 range)
#     return np.array(scores, dtype=np.float32), auc


# # ── main statistical validation function ────────────────────
# def statistical_validation(
#         test_iterator, gradcam_explainer, scorecam_explainer,
#         model, steps=30, baseline='blur',
#         max_images=None, alpha=0.05, plot=True):
#     """
#     Runs Deletion & Insertion evaluation on both explainers then
#     reports full statistical comparison the reviewer asked for:

#       • Mean ± SD per metric
#       • 95 % confidence intervals (t-distribution)
#       • Paired t-test  (parametric)
#       • Wilcoxon signed-rank test  (non-parametric)
#       • Cohen's d effect size
#       • ΔInsertion−Deletion score (faithfulness proxy)
#       • Summary bar chart + curve plot
#     """

#     def _get_cam(expl, image):
#         out = expl.predict_and_explain(image)
#         for key in ('gradcam', 'scorecam', 'cam'):
#             if key in out:
#                 return upsample_cam(np.array(out[key]), image.shape)
#         raise ValueError(f"{expl.__class__.__name__} returned no heatmap key")

#     records = []   # one dict per image

#     for batch_idx in tqdm(range(len(test_iterator)), desc="Evaluating"):
#         images_batch, _ = test_iterator[batch_idx]
#         for i in range(images_batch.shape[0]):
#             img = ensure_float01(np.array(images_batch[i], dtype=np.float32))
#             preds   = model.predict(img[np.newaxis], verbose=0)[0]
#             pred_idx = int(np.argmax(preds))

#             try:
#                 cam_g = _get_cam(gradcam_explainer, img)
#                 cam_s = _get_cam(scorecam_explainer, img)
#             except Exception as e:
#                 continue   # skip broken images silently

#             g_del_c, g_del_auc = deletion_insertion(model, img, cam_g, pred_idx, steps, 'deletion',  baseline)
#             g_ins_c, g_ins_auc = deletion_insertion(model, img, cam_g, pred_idx, steps, 'insertion', baseline)
#             s_del_c, s_del_auc = deletion_insertion(model, img, cam_s, pred_idx, steps, 'deletion',  baseline)
#             s_ins_c, s_ins_auc = deletion_insertion(model, img, cam_s, pred_idx, steps, 'insertion', baseline)

#             records.append(dict(
#                 g_del=g_del_auc, g_ins=g_ins_auc,
#                 s_del=s_del_auc, s_ins=s_ins_auc,
#                 g_del_curve=g_del_c, g_ins_curve=g_ins_c,
#                 s_del_curve=s_del_c, s_ins_curve=s_ins_c,
#             ))

#             if max_images and len(records) >= max_images:
#                 break
#         if max_images and len(records) >= max_images:
#             break

#     n = len(records)
#     if n < 2:
#         print("❌ Not enough images evaluated for statistics.")
#         return None

#     # ── collect arrays ───────────────────────────────────────
#     g_del = np.array([r['g_del'] for r in records])
#     g_ins = np.array([r['g_ins'] for r in records])
#     s_del = np.array([r['s_del'] for r in records])
#     s_ins = np.array([r['s_ins'] for r in records])

#     # ΔInsertion−Deletion  (higher = more faithful)
#     g_delta = g_ins - g_del
#     s_delta = s_ins - s_del

#     # ── statistical tests helper ─────────────────────────────
#     def run_tests(a, b, label):
#         diff = a - b
#         t_stat, t_p   = stats.ttest_rel(a, b)
#         w_stat, w_p   = stats.wilcoxon(diff) if np.any(diff != 0) else (np.nan, np.nan)
#         cohens_d      = diff.mean() / (diff.std(ddof=1) + 1e-10)
#         ci95          = stats.t.interval(0.95, df=n-1,
#                                          loc=diff.mean(),
#                                          scale=stats.sem(diff))
#         sig_t = "✓ sig" if t_p < alpha else "✗ n.s."
#         sig_w = "✓ sig" if (not np.isnan(w_p) and w_p < alpha) else "✗ n.s."
#         print(f"\n  [{label}]")
#         print(f"    GradCAM++  : {a.mean():.4f} ± {a.std():.4f}  "
#               f"  95% CI [{stats.t.interval(0.95,df=n-1,loc=a.mean(),scale=stats.sem(a))[0]:.4f}, "
#               f"{stats.t.interval(0.95,df=n-1,loc=a.mean(),scale=stats.sem(a))[1]:.4f}]")
#         print(f"    ScoreCAM   : {b.mean():.4f} ± {b.std():.4f}  "
#               f"  95% CI [{stats.t.interval(0.95,df=n-1,loc=b.mean(),scale=stats.sem(b))[0]:.4f}, "
#               f"{stats.t.interval(0.95,df=n-1,loc=b.mean(),scale=stats.sem(b))[1]:.4f}]")
#         print(f"    Paired t   : t={t_stat:+.3f}, p={t_p:.4f}  {sig_t}")
#         print(f"    Wilcoxon   : W={w_stat},    p={w_p:.4f}  {sig_w}")
#         print(f"    Cohen's d  : {cohens_d:+.3f}   (diff 95% CI [{ci95[0]:.4f}, {ci95[1]:.4f}])")
#         return dict(g_mean=a.mean(), g_std=a.std(), s_mean=b.mean(), s_std=b.std(),
#                     t_stat=t_stat, t_p=t_p, w_stat=w_stat, w_p=w_p,
#                     cohens_d=cohens_d, ci95_low=ci95[0], ci95_high=ci95[1])

#     print(f"\n{'='*60}")
#     print(f"  Statistical Validation  (n={n} images, α={alpha})")
#     print(f"{'='*60}")
#     r_del   = run_tests(g_del,   s_del,   "Deletion  AUC  (↓ better)")
#     r_ins   = run_tests(g_ins,   s_ins,   "Insertion AUC  (↑ better)")
#     r_delta = run_tests(g_delta, s_delta, "Δ Ins−Del      (↑ better)")
#     print(f"{'='*60}\n")

#     # ── plots ────────────────────────────────────────────────
#     if plot:
#         xs = np.linspace(0, 1, steps + 1)

#         g_del_m = np.mean([r['g_del_curve'] for r in records], axis=0)
#         g_del_s = np.std( [r['g_del_curve'] for r in records], axis=0)
#         g_ins_m = np.mean([r['g_ins_curve'] for r in records], axis=0)
#         g_ins_s = np.std( [r['g_ins_curve'] for r in records], axis=0)
#         s_del_m = np.mean([r['s_del_curve'] for r in records], axis=0)
#         s_del_s = np.std( [r['s_del_curve'] for r in records], axis=0)
#         s_ins_m = np.mean([r['s_ins_curve'] for r in records], axis=0)
#         s_ins_s = np.std( [r['s_ins_curve'] for r in records], axis=0)

#         fig = plt.figure(figsize=(16, 10))
#         gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

#         # ── (A) Deletion curves ──────────────────────────────
#         ax0 = fig.add_subplot(gs[0, :2])
#         ax0.plot(xs, g_del_m, color='tab:red',   label=f'GradCAM++ (AUC={r_del["g_mean"]:.3f})')
#         ax0.fill_between(xs, g_del_m-g_del_s, g_del_m+g_del_s, alpha=0.2, color='tab:red')
#         ax0.plot(xs, s_del_m, color='tab:blue',  label=f'ScoreCAM  (AUC={r_del["s_mean"]:.3f})', ls='--')
#         ax0.fill_between(xs, s_del_m-s_del_s, s_del_m+s_del_s, alpha=0.15, color='tab:blue')
#         ax0.set_title('(A) Deletion Curve  [↓ better]', fontweight='bold')
#         ax0.set_xlabel('Fraction of pixels removed'); ax0.set_ylabel('Predicted score')
#         ax0.legend(); ax0.grid(alpha=0.3)

#         # ── (B) Insertion curves ─────────────────────────────
#         ax1 = fig.add_subplot(gs[1, :2])
#         ax1.plot(xs, g_ins_m, color='tab:green',  label=f'GradCAM++ (AUC={r_ins["g_mean"]:.3f})')
#         ax1.fill_between(xs, g_ins_m-g_ins_s, g_ins_m+g_ins_s, alpha=0.2, color='tab:green')
#         ax1.plot(xs, s_ins_m, color='darkorchid', label=f'ScoreCAM  (AUC={r_ins["s_mean"]:.3f})', ls='--')
#         ax1.fill_between(xs, s_ins_m-s_ins_s, s_ins_m+s_ins_s, alpha=0.15, color='darkorchid')
#         ax1.set_title('(B) Insertion Curve  [↑ better]', fontweight='bold')
#         ax1.set_xlabel('Fraction of pixels added'); ax1.set_ylabel('Predicted score')
#         ax1.legend(); ax1.grid(alpha=0.3)

#         # ── (C) Bar chart with CI error bars ─────────────────
#         ax2 = fig.add_subplot(gs[0, 2])
#         metrics   = ['Deletion\nAUC', 'Insertion\nAUC', 'Δ Ins−Del']
#         g_vals    = [r_del['g_mean'],   r_ins['g_mean'],   g_delta.mean()]
#         s_vals    = [r_del['s_mean'],   r_ins['s_mean'],   s_delta.mean()]
#         g_err     = [r_del['g_std'],    r_ins['g_std'],    g_delta.std()]
#         s_err     = [r_del['s_std'],    r_ins['s_std'],    s_delta.std()]
#         x_pos     = np.arange(len(metrics))
#         w         = 0.35
#         ax2.bar(x_pos-w/2, g_vals, w, yerr=g_err, capsize=5,
#                 color='steelblue',  label='GradCAM++', alpha=0.85)
#         ax2.bar(x_pos+w/2, s_vals, w, yerr=s_err, capsize=5,
#                 color='tomato',     label='ScoreCAM',  alpha=0.85)
#         ax2.set_xticks(x_pos); ax2.set_xticklabels(metrics, fontsize=9)
#         ax2.set_title('(C) Mean ± SD', fontweight='bold')
#         ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)

#         # ── (D) Box plots ─────────────────────────────────────
#         ax3 = fig.add_subplot(gs[1, 2])
#         data_pairs = [g_del, s_del, g_ins, s_ins]
#         bp = ax3.boxplot(data_pairs, patch_artist=True,
#                          medianprops=dict(color='black', linewidth=2))
#         colors_bp = ['steelblue','tomato','mediumseagreen','darkorchid']
#         for patch, c in zip(bp['boxes'], colors_bp):
#             patch.set_facecolor(c); patch.set_alpha(0.7)
#         ax3.set_xticks([1,2,3,4])
#         ax3.set_xticklabels(['G++ Del','SC Del','G++ Ins','SC Ins'], fontsize=8)
#         ax3.set_title('(D) Distribution', fontweight='bold')
#         ax3.grid(axis='y', alpha=0.3)

#         # significance stars on box plot
#         def _sig_star(p):
#             if   p < 0.001: return '***'
#             elif p < 0.01:  return '**'
#             elif p < 0.05:  return '*'
#             else:           return 'n.s.'

#         y_top = max(max(g_del), max(s_del), max(g_ins), max(s_ins)) * 1.08
#         for pair, pval, xpair in [
#             ((1,2), r_del['t_p'],  (1,2)),
#             ((3,4), r_ins['t_p'],  (3,4))]:
#             x1, x2 = xpair
#             ax3.plot([x1, x1, x2, x2],
#                      [y_top, y_top*1.02, y_top*1.02, y_top], lw=1, color='k')
#             ax3.text((x1+x2)/2, y_top*1.025, _sig_star(pval),
#                      ha='center', va='bottom', fontsize=10)

#         fig.suptitle('Explainability Statistical Validation: GradCAM++ vs ScoreCAM',
#                      fontsize=13, fontweight='bold', y=1.01)
#         plt.show()

#     summary = dict(n_images=n,
#                    deletion=r_del, insertion=r_ins, delta=r_delta,
#                    raw=dict(g_del=g_del, g_ins=g_ins, s_del=s_del, s_ins=s_ins))
#     return summary


In [ ]:
# # ── Run it ───────────────────────────────────────────────────
# gradcam_explainer = GradCAMPlusPlus(model, 'conv2d_23', labels)
# scorecam_explainer = ScoreCAM(model, 'conv2d_23', labels)

# summary = statistical_validation(
#     test_iterator  = test_data,
#     gradcam_explainer  = gradcam_explainer,
#     scorecam_explainer = scorecam_explainer,
#     model     = model,
#     steps     = 30,
#     baseline  = 'blur',
#     max_images= 50,   
#     alpha     = 0.05,
#     plot      = True,
# )

In [ ]:
# ================================================================
#  HELPERS
# ================================================================
def ensure_float01(img):
    img = np.array(img, dtype=np.float32)
    return img / 255.0 if img.max() > 1.0 else img

def upsample_cam(cam, target_shape):
    cam_tf = tf.image.resize(cam[..., np.newaxis],
                             target_shape[:2], method='bilinear').numpy()[..., 0]
    mn, mx = cam_tf.min(), cam_tf.max()
    return (cam_tf - mn) / (mx - mn + 1e-8) if mx > mn else np.zeros_like(cam_tf)


# ================================================================
#  BASELINE DIAGNOSIS  — run this first to understand your model
# ================================================================
def diagnose_baseline(model, test_iterator, labels, n=5):
    print("=" * 55)
    print("  BASELINE DIAGNOSIS")
    print("=" * 55)
    for b in range(min(n, len(test_iterator))):
        img      = ensure_float01(np.array(test_iterator[b][0][0], dtype=np.float32))
        orig_s   = model.predict(img[np.newaxis],                    verbose=0)[0]
        blur_s   = model.predict(gaussian_filter(img, sigma=(21,21,0))[np.newaxis], verbose=0)[0]
        zero_s   = model.predict(np.zeros_like(img)[np.newaxis],     verbose=0)[0]
        mean_s   = model.predict(np.full_like(img, 0.5)[np.newaxis], verbose=0)[0]
        pred_idx = int(np.argmax(orig_s))
        print(f"\n  Image {b} → predicted: {labels[pred_idx]}")
        print(f"    Original  : {orig_s[pred_idx]:.4f}")
        print(f"    Blur base : {blur_s[pred_idx]:.4f}  ← should be LOW")
        print(f"    Zero base : {zero_s[pred_idx]:.4f}  ← should be LOW")
        print(f"    Mean base : {mean_s[pred_idx]:.4f}  ← should be LOW")
        best_base_conf = min(blur_s[pred_idx], zero_s[pred_idx], mean_s[pred_idx])
        score_range    = orig_s[pred_idx] - best_base_conf
        print(f"    Best achievable score_range: {score_range:.4f}", end="  ")
        if score_range < 0.05:
            print("⚠️  MODEL TOO OVERCONFIDENT — normalized metrics will be used")
        else:
            print("✓ OK")
    print("=" * 55)


# ================================================================
#  CORE METRIC — normalized deletion / insertion
#  Handles overconfident models by measuring RELATIVE change:
#      norm_score = (raw - baseline_conf) / (orig_conf - baseline_conf)
#  → always starts near 1.0 and ends near 0.0 regardless of
#    absolute confidence values
# ================================================================
def deletion_insertion(model, image, cam, class_idx, steps=30, mode='deletion'):
    img     = ensure_float01(image)
    H, W, C = img.shape

    # --- pick best baseline (lowest confidence = most informative) ---
    candidates = {
        'blur' : gaussian_filter(img, sigma=(21, 21, 0)),
        'zero' : np.zeros_like(img),
        'mean' : np.full_like(img, 0.5),
    }
    base_confs = {
        name: float(model.predict(b[np.newaxis], verbose=0)[0][class_idx])
        for name, b in candidates.items()
    }
    best_name = min(base_confs, key=base_confs.get)
    base      = candidates[best_name]
    base_conf = base_confs[best_name]
    orig_conf = float(model.predict(img[np.newaxis], verbose=0)[0][class_idx])
    score_range = orig_conf - base_conf   # may be near 0 for overconfident models

    order   = np.argsort(-cam.flatten())   # most → least salient pixel
    step_px = max(1, H * W // steps)
    raw     = []

    cur = img.copy() if mode == 'deletion' else base.copy()
    raw.append(float(model.predict(cur[np.newaxis], verbose=0)[0][class_idx]))

    for s in range(1, steps + 1):
        k      = min(s * step_px, H * W)
        idx    = order[:k]
        ys, xs = idx // W, idx % W
        if mode == 'deletion':
            cur[ys, xs, :] = base[ys, xs, :]
        else:
            cur[ys, xs, :] = img[ys, xs, :]
        raw.append(float(model.predict(cur[np.newaxis], verbose=0)[0][class_idx]))

    raw = np.array(raw, dtype=np.float32)

    # normalize so that 1.0 = original confidence, 0.0 = baseline confidence
    if abs(score_range) > 1e-4:
        norm = (raw - base_conf) / (score_range + 1e-8)
        norm = np.clip(norm, -0.5, 1.5)
    else:
        # truly flat model — assign 0.5 (uninformative, honestly reported)
        norm = np.full_like(raw, 0.5)

    xs_axis = np.linspace(0, 1, len(norm))
    auc     = float(np.trapz(norm, xs_axis))
    return norm, auc, best_name, orig_conf, base_conf


# ================================================================
#  STATISTICAL VALIDATION
# ================================================================
def statistical_validation(
        test_iterator,
        gradcam_explainer,
        scorecam_explainer,
        model,
        steps     = 30,
        max_images= None,
        alpha     = 0.05,
        plot      = True):
    """
    Full statistical comparison of GradCAM++ vs ScoreCAM using
    normalized Deletion & Insertion AUC.

    Handles overconfident models automatically via per-image
    baseline normalization.

    Reports:
      • Mean ± SD  with 95% confidence intervals
      • Paired t-test  (parametric)
      • Wilcoxon signed-rank test  (non-parametric)
      • Cohen's d  effect size
      • Δ Insertion − Deletion  (faithfulness proxy)
      • 4-panel plot
    """

    def _get_cam(expl, image):
        out = expl.predict_and_explain(image)
        for key in ('gradcam', 'scorecam', 'cam'):
            if key in out:
                return upsample_cam(np.array(out[key]), image.shape)
        raise ValueError(f"{expl.__class__.__name__} returned no heatmap key")

    records    = []
    flat_count = 0

    for batch_idx in tqdm(range(len(test_iterator)), desc="Evaluating"):
        images_batch, _ = test_iterator[batch_idx]
        for i in range(images_batch.shape[0]):
            img      = ensure_float01(np.array(images_batch[i], dtype=np.float32))
            preds    = model.predict(img[np.newaxis], verbose=0)[0]
            pred_idx = int(np.argmax(preds))

            if float(preds[pred_idx]) < 0.5:
                continue   # skip uncertain predictions

            try:
                cam_g = _get_cam(gradcam_explainer, img)
                cam_s = _get_cam(scorecam_explainer, img)
            except Exception:
                continue

            g_del_c, g_del_auc, bl, oc, bc = deletion_insertion(
                model, img, cam_g, pred_idx, steps, 'deletion')
            g_ins_c, g_ins_auc, _,  _,  _  = deletion_insertion(
                model, img, cam_g, pred_idx, steps, 'insertion')
            s_del_c, s_del_auc, _,  _,  _  = deletion_insertion(
                model, img, cam_s, pred_idx, steps, 'deletion')
            s_ins_c, s_ins_auc, _,  _,  _  = deletion_insertion(
                model, img, cam_s, pred_idx, steps, 'insertion')

            if (oc - bc) < 1e-4:
                flat_count += 1

            records.append(dict(
                g_del=g_del_auc, g_ins=g_ins_auc,
                s_del=s_del_auc, s_ins=s_ins_auc,
                g_del_curve=g_del_c, g_ins_curve=g_ins_c,
                s_del_curve=s_del_c, s_ins_curve=s_ins_c,
                orig_conf=oc, base_conf=bc, baseline_used=bl,
            ))

            if max_images and len(records) >= max_images:
                break
        if max_images and len(records) >= max_images:
            break

    n = len(records)
    if n < 2:
        print("❌ Not enough valid images evaluated.")
        return None

    if flat_count > 0:
        print(f"\n⚠️  {flat_count}/{n} images had score_range < 0.0001 "
              f"(model equally confident on baseline & original). "
              f"Those images contribute AUC=0.5 (uninformative) to both methods equally.")

    # ── collect arrays ──────────────────────────────────────
    g_del   = np.array([r['g_del'] for r in records])
    g_ins   = np.array([r['g_ins'] for r in records])
    s_del   = np.array([r['s_del'] for r in records])
    s_ins   = np.array([r['s_ins'] for r in records])
    g_delta = g_ins - g_del
    s_delta = s_ins - s_del

    # ── stats helper ────────────────────────────────────────
    def run_tests(a, b, label):
        diff        = a - b
        t_stat, t_p = stats.ttest_rel(a, b)
        try:
            w_stat, w_p = stats.wilcoxon(diff)
        except Exception:
            w_stat, w_p = np.nan, np.nan
        cohens_d = diff.mean() / (diff.std(ddof=1) + 1e-10)
        ci_diff  = stats.t.interval(0.95, df=n-1, loc=diff.mean(), scale=stats.sem(diff))
        ci_a     = stats.t.interval(0.95, df=n-1, loc=a.mean(),    scale=stats.sem(a))
        ci_b     = stats.t.interval(0.95, df=n-1, loc=b.mean(),    scale=stats.sem(b))
        sig_t    = "✓ sig" if t_p < alpha else "✗ n.s."
        sig_w    = "✓ sig" if (not np.isnan(w_p) and w_p < alpha) else "✗ n.s."
        print(f"\n  [{label}]")
        print(f"    GradCAM++  : {a.mean():.4f} ± {a.std():.4f}   "
              f"95% CI [{ci_a[0]:.4f}, {ci_a[1]:.4f}]")
        print(f"    ScoreCAM   : {b.mean():.4f} ± {b.std():.4f}   "
              f"95% CI [{ci_b[0]:.4f}, {ci_b[1]:.4f}]")
        print(f"    Paired t   : t={t_stat:+.3f}, p={t_p:.4f}  {sig_t}")
        print(f"    Wilcoxon   : W={w_stat},    p={w_p:.4f}  {sig_w}")
        print(f"    Cohen's d  : {cohens_d:+.3f}   "
              f"(diff 95% CI [{ci_diff[0]:.4f}, {ci_diff[1]:.4f}])")
        return dict(
            g_mean=float(a.mean()), g_std=float(a.std()),
            s_mean=float(b.mean()), s_std=float(b.std()),
            t_stat=float(t_stat),   t_p=float(t_p),
            w_stat=w_stat,          w_p=float(w_p),
            cohens_d=float(cohens_d),
            ci_diff_low=float(ci_diff[0]), ci_diff_high=float(ci_diff[1]))

    avg_orig = np.mean([r['orig_conf'] for r in records])
    avg_base = np.mean([r['base_conf'] for r in records])
    baseline_names = [r['baseline_used'] for r in records]
    most_common_bl = max(set(baseline_names), key=baseline_names.count)

    print(f"\n{'='*60}")
    print(f"  Statistical Validation  (n={n} images, α={alpha})")
    print(f"  Most-used baseline      : {most_common_bl}")
    print(f"  Avg original confidence : {avg_orig:.4f}")
    print(f"  Avg baseline confidence : {avg_base:.4f}   "
          f"(score range = {avg_orig-avg_base:.4f})")
    print(f"{'='*60}")

    r_del   = run_tests(g_del,   s_del,   "Deletion  AUC  (↓ better)")
    r_ins   = run_tests(g_ins,   s_ins,   "Insertion AUC  (↑ better)")
    r_delta = run_tests(g_delta, s_delta, "Δ Ins−Del      (↑ better)")
    print(f"{'='*60}\n")

    # ── plots ───────────────────────────────────────────────
    if plot:
        max_len = max(len(r['g_del_curve']) for r in records)

        def pad(curves):
            return np.array([
                np.pad(c, (0, max_len - len(c)), 'edge') for c in curves
            ])

        g_del_m = np.mean(pad([r['g_del_curve'] for r in records]), axis=0)
        g_del_s = np.std( pad([r['g_del_curve'] for r in records]), axis=0)
        g_ins_m = np.mean(pad([r['g_ins_curve'] for r in records]), axis=0)
        g_ins_s = np.std( pad([r['g_ins_curve'] for r in records]), axis=0)
        s_del_m = np.mean(pad([r['s_del_curve'] for r in records]), axis=0)
        s_del_s = np.std( pad([r['s_del_curve'] for r in records]), axis=0)
        s_ins_m = np.mean(pad([r['s_ins_curve'] for r in records]), axis=0)
        s_ins_s = np.std( pad([r['s_ins_curve'] for r in records]), axis=0)
        xs_plot = np.linspace(0, 1, max_len)

        fig = plt.figure(figsize=(16, 10))
        gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

        # (A) Deletion curves
        ax0 = fig.add_subplot(gs[0, :2])
        ax0.plot(xs_plot, g_del_m, color='tab:red',
                 label=f'GradCAM++ (AUC={r_del["g_mean"]:.3f})')
        ax0.fill_between(xs_plot, g_del_m-g_del_s, g_del_m+g_del_s,
                         alpha=0.2, color='tab:red')
        ax0.plot(xs_plot, s_del_m, color='tab:blue', ls='--',
                 label=f'ScoreCAM  (AUC={r_del["s_mean"]:.3f})')
        ax0.fill_between(xs_plot, s_del_m-s_del_s, s_del_m+s_del_s,
                         alpha=0.15, color='tab:blue')
        ax0.axhline(0, color='gray', lw=0.8, ls=':')
        ax0.axhline(1, color='gray', lw=0.8, ls=':')
        ax0.set_title('(A) Deletion Curve  [↓ AUC = better]', fontweight='bold')
        ax0.set_xlabel('Fraction of pixels removed')
        ax0.set_ylabel('Normalised predicted score')
        ax0.set_ylim(-0.1, 1.15)
        ax0.legend(); ax0.grid(alpha=0.3)

        # (B) Insertion curves
        ax1 = fig.add_subplot(gs[1, :2])
        ax1.plot(xs_plot, g_ins_m, color='tab:green',
                 label=f'GradCAM++ (AUC={r_ins["g_mean"]:.3f})')
        ax1.fill_between(xs_plot, g_ins_m-g_ins_s, g_ins_m+g_ins_s,
                         alpha=0.2, color='tab:green')
        ax1.plot(xs_plot, s_ins_m, color='darkorchid', ls='--',
                 label=f'ScoreCAM  (AUC={r_ins["s_mean"]:.3f})')
        ax1.fill_between(xs_plot, s_ins_m-s_ins_s, s_ins_m+s_ins_s,
                         alpha=0.15, color='darkorchid')
        ax1.axhline(0, color='gray', lw=0.8, ls=':')
        ax1.axhline(1, color='gray', lw=0.8, ls=':')
        ax1.set_title('(B) Insertion Curve  [↑ AUC = better]', fontweight='bold')
        ax1.set_xlabel('Fraction of pixels added')
        ax1.set_ylabel('Normalised predicted score')
        ax1.set_ylim(-0.1, 1.15)
        ax1.legend(); ax1.grid(alpha=0.3)

        # (C) Bar chart with SD error bars
        ax2 = fig.add_subplot(gs[0, 2])
        metrics = ['Deletion\nAUC', 'Insertion\nAUC', 'Δ Ins−Del']
        g_vals  = [r_del['g_mean'],  r_ins['g_mean'],  float(g_delta.mean())]
        s_vals  = [r_del['s_mean'],  r_ins['s_mean'],  float(s_delta.mean())]
        g_err   = [r_del['g_std'],   r_ins['g_std'],   float(g_delta.std())]
        s_err   = [r_del['s_std'],   r_ins['s_std'],   float(s_delta.std())]
        x_pos   = np.arange(len(metrics))
        w       = 0.35
        ax2.bar(x_pos - w/2, g_vals, w, yerr=g_err, capsize=5,
                color='steelblue', label='GradCAM++', alpha=0.85)
        ax2.bar(x_pos + w/2, s_vals, w, yerr=s_err, capsize=5,
                color='tomato',    label='ScoreCAM',  alpha=0.85)
        ax2.set_xticks(x_pos); ax2.set_xticklabels(metrics, fontsize=9)
        ax2.set_title('(C) Mean ± SD', fontweight='bold')
        ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)

        # (D) Boxplots + significance stars
        ax3 = fig.add_subplot(gs[1, 2])
        bp = ax3.boxplot(
            [g_del, s_del, g_ins, s_ins],
            patch_artist=True,
            medianprops=dict(color='black', linewidth=2)
        )
        for patch, c in zip(bp['boxes'],
                            ['steelblue', 'tomato', 'mediumseagreen', 'darkorchid']):
            patch.set_facecolor(c); patch.set_alpha(0.7)
        ax3.set_xticks([1, 2, 3, 4])
        ax3.set_xticklabels(['G++ Del', 'SC Del', 'G++ Ins', 'SC Ins'], fontsize=8)
        ax3.set_title('(D) Distribution', fontweight='bold')
        ax3.grid(axis='y', alpha=0.3)

        def sig_star(p):
            if   p < 0.001: return '***'
            elif p < 0.01:  return '**'
            elif p < 0.05:  return '*'
            else:           return 'n.s.'

        all_vals = np.concatenate([g_del, s_del, g_ins, s_ins])
        y_top    = all_vals.max() * 1.08
        for (x1, x2), pval in [((1, 2), r_del['t_p']), ((3, 4), r_ins['t_p'])]:
            ax3.plot([x1, x1, x2, x2],
                     [y_top, y_top*1.03, y_top*1.03, y_top], lw=1, color='k')
            ax3.text((x1+x2)/2, y_top*1.035, sig_star(pval),
                     ha='center', va='bottom', fontsize=10)

        fig.suptitle(
            f'Explainability Statistical Validation: GradCAM++ vs ScoreCAM  '
            f'(n={n}, normalised scores)',
            fontsize=13, fontweight='bold', y=1.01)
        plt.show()

    return dict(n_images=n,
                deletion=r_del, insertion=r_ins, delta=r_delta,
                raw=dict(g_del=g_del, g_ins=g_ins, s_del=s_del, s_ins=s_ins))


# ================================================================
#  RUN
# ================================================================
gradcam_explainer  = GradCAMPlusPlus(model, 'conv2d_23', labels)
scorecam_explainer = ScoreCAM(model, 'conv2d_23', labels)

# Step 1 — always diagnose first
diagnose_baseline(model, test_data, labels, n=5)

# Step 2 — run full validation
summary = statistical_validation(
    test_iterator      = test_data,
    gradcam_explainer  = gradcam_explainer,
    scorecam_explainer = scorecam_explainer,
    model              = model,
    steps              = 30,
    max_images         = 100,
    alpha              = 0.05,
    plot               = True,
)